## 1. Подготовка

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')

TOKEN = userdata.get("GH_TOKEN")
REPO = f"https://{TOKEN}@github.com/BEknUur/huawei-ai-final-project.git"
AUDIO = "/content/drive/MyDrive/project22_audio"
os.makedirs(AUDIO, exist_ok=True)

!git clone -q {REPO} /content/project22
%cd /content/project22
!rm -rf data/audio && ln -s {AUDIO} data/audio
!pip install -q -r requirements.txt
!git clone -q https://github.com/BUTSpeechFIT/AMI-diarization-setup /content/ami

## 2. Митинги

In [ ]:
DEV  = ["ES2011a", "IS1008a", "TS3004a"]
TEST = ["EN2002a", "EN2002b", "EN2002c", "EN2002d",
        "ES2004a", "ES2004b", "ES2004c", "ES2004d",
        "IS1009a", "IS1009b", "IS1009c", "IS1009d",
        "TS3003a", "TS3003b", "TS3003c", "TS3003d"]

def get_data(meetings, split):
    for m in meetings:
        !cp /content/ami/only_words/rttms/{split}/{m}.rttm data/ref_rttm/
        !cp /content/ami/uems/{split}/{m}.uem data/uem/
        if not os.path.exists(f"data/audio/{m}.Mix-Headset.wav"):
            !wget -q -O data/audio/{m}.Mix-Headset.wav https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/{m}/audio/{m}.Mix-Headset.wav
        print(m, "ok")

get_data(DEV, "dev")

## 3. Модель

In [ ]:
import torch, time, soundfile as sf
from pyannote.audio import Pipeline
from pyannote.metrics.diarization import DiarizationErrorRate
from diarization import load_rttm, load_uem

pipe = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
                                use_auth_token=userdata.get("HF_TOKEN"))
pipe.to(torch.device("cuda"))

def run(m, save_to=None):
    wav, sr = sf.read(f"data/audio/{m}.Mix-Headset.wav", dtype="float32")
    start = time.time()
    hyp = pipe({"uri": m, "waveform": torch.tensor(wav).unsqueeze(0), "sample_rate": sr})
    sec = time.time() - start
    if save_to:
        with open(f"{save_to}/{m}.rttm", "w") as f:
            hyp.write_rttm(f)
    return hyp, sec

def ref_uem(m):
    return load_rttm(f"data/ref_rttm/{m}.rttm")[m], load_uem(f"data/uem/{m}.uem")[m]

## 4. Спринт 1 — один митинг

In [ ]:
m = DEV[0]
hyp, sec = run(m)
ref, uem = ref_uem(m)
der = DiarizationErrorRate(collar=0.0, skip_overlap=False)
print(f"{m}: DER = {der(ref, hyp, uem=uem):.3f}")
print(f"спикеров: эталон {len(ref.labels())}, pyannote {len(hyp.labels())}, время {sec:.0f} c")

## 5. Спринт 2 — подбор порога на dev

In [ ]:
import copy, pandas as pd
default = pipe.parameters(instantiated=True)
print("дефолтный порог:", default["clustering"]["threshold"])

results = []
for thr in [0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
    params = copy.deepcopy(default)
    params["clustering"]["threshold"] = thr
    pipe.instantiate(params)
    der = DiarizationErrorRate(collar=0.0, skip_overlap=False)
    for m in DEV:
        hyp, _ = run(m)
        ref, uem = ref_uem(m)
        der(ref, hyp, uem=uem)
    results.append({"threshold": thr, "dev_DER": round(abs(der), 4)})
    print(results[-1])

tuning = pd.DataFrame(results)
tuning.to_csv("results/tuning_pyannote.csv", index=False)
BEST = tuning.loc[tuning.dev_DER.idxmin(), "threshold"]
print("лучший порог:", BEST)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(tuning.threshold, tuning.dev_DER * 100, marker="o")
plt.xlabel("clustering threshold"); plt.ylabel("DER на dev, %"); plt.grid(alpha=.3)
plt.savefig("figures/pyannote_tuning.png", dpi=200); plt.show()

## 6. Спринт 3 — test (16 митингов)

In [ ]:
get_data(TEST, "test")

params = copy.deepcopy(default)
params["clustering"]["threshold"] = BEST
pipe.instantiate(params)

times = []
for m in TEST:
    hyp, sec = run(m, save_to="outputs/pyannote")
    times.append({"meeting": m, "seconds": round(sec, 1), "speakers": len(hyp.labels())})
    print(times[-1])
pd.DataFrame(times).to_csv("results/runtime_pyannote.csv", index=False)

In [ ]:
der = DiarizationErrorRate(collar=0.0, skip_overlap=False)
for m in TEST:
    ref, uem = ref_uem(m)
    der(ref, load_rttm(f"outputs/pyannote/{m}.rttm")[m], uem=uem)
print(f"test DER: {abs(der):.3f}")

## 7. Сохранить в GitHub

In [ ]:
!git config user.name "Beknur" && git config user.email "ualihanulybeknur@gmail.com"
!git add outputs/pyannote results figures data/ref_rttm data/uem
!git commit -q -m "pyannote: tuning + test RTTM"
!git pull -q --rebase {REPO} main && git push -q {REPO} main
print("готово")